# **1. Setup**



In [1]:
!pip install transformers datasets torch scikit-learn -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import numpy as np
from datasets import load_dataset, Dataset as HFDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# **2. Data Acquisition**



In [2]:
dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

# Subsample for faster experiments
dataset = dataset.shuffle(seed=42).select(range(1000))

set(dataset['Category of Hallucination'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'Incomplete Information',
 'Mechanism and Pathway Misattribution',
 'Methodological and Evidence Fabrication',
 'Misinterpretation of #Question#'}

In [3]:
# Look at the first few examples
for i in range(4):
  print("Question: ", dataset['Question'][i])
  print("Knowledge: ", dataset['Knowledge'][i])
  print("Ground Truth: ", dataset['Ground Truth'][i])
  print("Hallucinated Answer: ", dataset['Hallucinated Answer'][i])
  print("Category of Hallucination: ", dataset['Category of Hallucination'][i])
  print()

Question:  Is eligibility for a chemotherapy protocol a good prognostic factor for invasive bladder cancer after radical cystectomy?
Knowledge:  ['To assess whether eligibility to an adjuvant chemotherapy protocol in itself represents a good prognostic factor after radical cystectomy for bladder cancer.', 'Between April 1984 and May 1989, our institution entered 35 patients with invasive bladder cancer into the Swiss Group for Clinical and Epidemiological Cancer Research (SAKK) study 09/84. They were randomly assigned to either observation or three postoperative courses of cisplatin monotherapy after cystectomy. This study had a negative result. The outcome of these 35 patients (protocol group) was compared with an age- and tumor-stage-matched cohort (matched group; n = 35) who also underwent cystectomy during the same period, but were not entered into the SAKK study, as well as the remaining 57 patients treated during the study period for the same indication (remaining group).', 'Medi

# **3. Preprocess dataset**

In [11]:
def preprocess(example):
    processed = []
    question = example["Question"]
    knowledge = " ".join(example["Knowledge"]) if isinstance(example["Knowledge"], list) else example["Knowledge"]
    gt_answer = example["Ground Truth"]
    hallucinated_answer = example["Hallucinated Answer"]

    if not gt_answer or not hallucinated_answer:
        return []

    premise = f"Question: {question} Context: {knowledge}"

    # Positive example (entailed)
    processed.append({"text_a": premise, "text_b": gt_answer, "label": 0})
    # Negative example (hallucinated)
    processed.append({"text_a": premise, "text_b": hallucinated_answer, "label": 1})

    return processed

processed_data = []
for ex in dataset:
    processed_data.extend(preprocess(ex))

# **4. Tokenization**

In [12]:
checkpoint = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)
max_length = 256

# Tokenize manually
input_ids, attention_masks, token_type_ids, labels = [], [], [], []

for ex in processed_data:
    encoded = tokenizer(
        ex["text_a"],
        ex["text_b"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids.append(encoded['input_ids'])
    attention_masks.append(encoded['attention_mask'])
    token_type_ids.append(encoded['token_type_ids'])
    labels.append(torch.tensor(ex['label']))

# Stack into tensors
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
token_type_ids = torch.cat(token_type_ids, dim=0)
labels = torch.stack(labels)

# shuffle
perm = torch.randperm(len(labels))

input_ids = input_ids[perm]
attention_masks = attention_masks[perm]
token_type_ids = token_type_ids[perm]
labels = labels[perm]

# **5. Build final dataset**

In [13]:
class HallucinationDataset(Dataset):
    def __init__(self, input_ids, attention_masks, token_type_ids, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.token_type_ids = token_type_ids
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "token_type_ids": self.token_type_ids[idx],
            "labels": self.labels[idx]
        }

# Train / validation split
split_idx = int(0.8 * len(labels))
train_dataset = HallucinationDataset(input_ids[:split_idx], attention_masks[:split_idx],
                                     token_type_ids[:split_idx], labels[:split_idx])
val_dataset = HallucinationDataset(input_ids[split_idx:], attention_masks[split_idx:],
                                   token_type_ids[split_idx:], labels[split_idx:])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# **6. BERT model & optimzer**

In [14]:
model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
for name, param in model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

# **7. Training and Evaluate**

In [15]:
epochs = 3

print("Epoch\tTrain Loss\tVal Loss\tAcc\tF1\tPrecision\tRecall")

for epoch in range(epochs):

    # TRAIN
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=batch_labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    model.eval()
    total_val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            batch_labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=batch_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_labels.extend(batch_labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )

    acc = np.mean(np.array(all_labels) == np.array(all_preds))

    print(f"{epoch+1}\t{avg_train_loss:.4f}\t\t{avg_val_loss:.4f}\t\t{acc:.4f}\t{f1:.4f}\t{precision:.4f}\t{recall:.4f}")

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)

Epoch	Train Loss	Val Loss	Acc	F1	Precision	Recall
1	0.4333		0.2296		0.9200	0.9259	0.9259	0.9259
Confusion Matrix:
 [[168  16]
 [ 16 200]]
2	0.1786		0.3767		0.8900	0.9060	0.8413	0.9815
Confusion Matrix:
 [[144  40]
 [  4 212]]
3	0.1002		0.2632		0.9050	0.9128	0.9045	0.9213
Confusion Matrix:
 [[163  21]
 [ 17 199]]
